In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("data/energydata_complete.csv", index_col="date",
                infer_datetime_format = True)

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df["Windspeed"].plot(figsize=(12,8));

In [ ]:
df["Appliances"].plot(figsize=(12,8));

**Train Test Split**

In [ ]:
len(df)

In [ ]:
df.head(3)

In [ ]:
df.tail()

In [ ]:
# Let's predict 24 hours -> we'll select the last month of data

In [ ]:
df_2 = df.loc["2016-05-01":] # Month of may

In [ ]:
df_2 = df_2.round(2) # 2 decimal point precision

In [ ]:
# How many rows per day?

24*60 / 10

In [ ]:
test_days = 2

In [ ]:
test_ind = test_days * 144

In [ ]:
train = df_2.iloc[:-test_ind]
test = df_2.iloc[-test_ind:]

In [ ]:
train.shape

In [ ]:
test.shape

**Scale the data**

In [ ]:
from sklearn.preprocessing import MinMaxScaler

In [ ]:
scaler = MinMaxScaler()
scaled_train = scaler.fit_transform(train)
scaled_test = scaler.transform(test)

**Time Series Generator**

In [ ]:
from tensorflow.keras.preprocessing.sequence import TimeseriesGenerator

In [ ]:
length = 144
batch_size = 1

generator = TimeseriesGenerator(scaled_train, scaled_train,
                               length=length, batch_size=batch_size)

In [ ]:
X, y = generator[0]

In [ ]:
# Given the last 144 points predict the next x
print("Given the array : \n{}".format(X.flatten()))
print("Predict this {}: \n".format(y))

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM

In [ ]:
scaled_train.shape

In [ ]:
model = Sequential()

model.add(LSTM(25, input_shape=(length, scaled_train.shape[1])))

model.add(Dense(scaled_train.shape[1]))

model.compile(optimizer="adam",
             loss="mse")

In [ ]:
model.summary()

**Early Stopping**

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
early_stop = EarlyStopping(monitor="val_loss", patience=2)

validation_generator = TimeseriesGenerator(scaled_test, scaled_test,
                                          length=length,
                                          batch_size=batch_size)

In [ ]:
#Takes a lot of time
model.fit_generator(generator, epochs=2,
                   validation_data = validation_generator,
                   callbacks=[early_stop])

**Losses**

In [ ]:
losses = pd.DataFrame(model.history.history)
losses.plot()

**Evaluation**

In [ ]:
n_features = scaled_train.shape[1]
test_predictions = []

first_eval_batch = scaled_train[-length:]
current_batch = first_eval_batch.reshape((1, length,
                                            scaled_train.shape[1]))

for i in range(len(test)):
    
    current_pred = model.predict(current_batch)[0]
    
    test_predictions.append(current_pred)
    
    current_batch = np.append(current_batch[:, 1:, :],
                             [[current_pred]], axis=1)

In [ ]:
true_predictions = scaler.inverse_transform(test_predictions)